# Single-system subhalo realization

Linear, single-process port of `mejiro/pipeline/_03_generate_subhalos.py` for one lens pickle. Set `sca`, `band`, and `config_path` below. (`band` is unused by this step, but kept for symmetry with the synthetic-image notebook.)

In [ ]:
sca = 1
band = 'F129'
config_path = '/grad/bwedig/mejiro/projects/roman_data_challenge/roman_data_challenge_rung_1.yaml'

# Optional: pick a specific lens pickle. If None, the first lens found in the SCA's _02 directory is used.
lens_pickle = None

In [ ]:
import os
from glob import glob

import numpy as np
import yaml
from pyHalo.preset_models import preset_model_from_name

from mejiro.utils import roman_util, util

In [ ]:
with open(config_path, 'r') as f:
    config = yaml.load(f, Loader=yaml.SafeLoader)

subhalo_config = config['subhalos']
use_jax = config['jaxtronomy']['use_jax']

data_dir = config['data_dir']
pipeline_label = config['pipeline_label']
if config.get('dev'):
    pipeline_label += '_dev'
pipeline_dir = os.path.join(data_dir, pipeline_label)
pipeline_dir

In [ ]:
sca_string = roman_util.get_sca_string(sca).lower()
input_dir = os.path.join(pipeline_dir, '02', sca_string)

if lens_pickle is None:
    matches = sorted(glob(os.path.join(input_dir, f'lens_{pipeline_label}_*.pkl')))
    assert matches, f'No lens pickles found in {input_dir}'
    lens_pickle = matches[0]

lens = util.unpickle(lens_pickle)
lens_pickle, lens.name

In [ ]:
realization_kwargs = dict(subhalo_config['realization_kwargs'])
if 'cone_opening_angle_arcsec' not in realization_kwargs:
    realization_kwargs['cone_opening_angle_arcsec'] = lens.get_einstein_radius() * 3

REALIZATION = preset_model_from_name(subhalo_config['pyhalo_model'])
realization = REALIZATION(
    z_lens=round(lens.z_lens, 2),
    z_source=round(lens.z_source, 2),
    log_m_host=np.log10(lens.get_main_halo_mass()),
    kwargs_cosmo=util.get_kwargs_cosmo(lens.cosmo),
    **realization_kwargs,
)
realization

In [ ]:
lens.add_realization(realization, use_jax=use_jax)
lens